In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

In [22]:
df = pd.read_csv('meteorite_landings.csv')
df.rename(columns={'mass (g)': 'mass'}, inplace=True)
df.head()
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45716 entries, 0 to 45715
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Name            45716 non-null  object
 1   ID              45716 non-null  int64 
 2   NameType        45716 non-null  object
 3   Classification  45716 non-null  object
 4   Mass            45716 non-null  object
 5   Fall            45716 non-null  object
 6   Year            45716 non-null  object
 7   Coordinates     45716 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.8+ MB


,Name,ID,NameType,Classification,Mass,Fall,Year,Coordinates
count,45716,45716.000000,45716,45716,45716,45716,45716,45716
unique,45716,NaN,2,466,12577,2,265,17100
top,Aachen,NaN,Valid,L6,"Quantity[1.3, ""Grams""]",Found,"DateObject[{2003}, ""Year"", ""Gregorian"", -5.]","Missing[""NotAvailable""]"
freq,1,NaN,45641,8285,171,44609,3323,13529
mean,NaN,26889.735104,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,16860.683030,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,12688.750000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,24261.500000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,40656.750000,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
df["mass_value"] = pd.to_numeric(
    df["Mass"].str.extract(r"Quantity\[(\d+\.?\d*),")[0], 
    errors="coerce"
    
)
df.head(10)

,Name,ID,NameType,Classification,Mass,Fall,Year,Coordinates,reclat,reclong,mass_value
0,Aachen,1,Valid,L5,"Quantity[21, ""Grams""]",Fell,"DateObject[{1880}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{50.775, 6.08333}]",50.77500,6.08333,21.0
1,Aarhus,2,Valid,H6,"Quantity[720, ""Grams""]",Fell,"DateObject[{1951}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{56.18333, 10.23333}]",56.18333,10.23333,720.0
2,Abee,6,Valid,EH4,"Quantity[107000, ""Grams""]",Fell,"DateObject[{1952}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{54.21667, -113.}]",54.21667,-113.00000,107000.0
3,Acapulco,10,Valid,Acapulcoite,"Quantity[1914, ""Grams""]",Fell,"DateObject[{1976}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{16.88333, -99.9}]",16.88333,-99.90000,1914.0
4,Achiras,370,Valid,L6,"Quantity[780, ""Grams""]",Fell,"DateObject[{1902}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{-33.16667, -64.95}]",-33.16667,-64.95000,780.0
5,Adhi Kot,379,Valid,EH4,"Quantity[4239, ""Grams""]",Fell,"DateObject[{1919}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{32.1, 71.8}]",32.10000,71.80000,4239.0
6,Adzhi-Bogdo (stone),390,Valid,LL3-6,"Quantity[910, ""Grams""]",Fell,"DateObject[{1949}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{44.83333, 95.16667}]",44.83333,95.16667,910.0
7,Agen,392,Valid,H5,"Quantity[30000, ""Grams""]",Fell,"DateObject[{1814}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{44.21667, 0.61667}]",44.21667,0.61667,30000.0
8,Aguada,398,Valid,L6,"Quantity[1620, ""Grams""]",Fell,"DateObject[{1930}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{-31.6, -65.23333}]",-31.60000,-65.23333,1620.0
9,Aguila Blanca,417,Valid,L,"Quantity[1440, ""Grams""]",Fell,"DateObject[{1920}, ""Year"", ""Gregorian"", -5.]","GeoPosition[{-30.86667, -64.55}]",-30.86667,-64.55000,1440.0


In [46]:
df["mass_value"] = pd.to_numeric(
    df["Mass"].str.extract(r"Quantity\[(\d+\.?\d*),")[0], 
    errors="coerce"
)

In [48]:
df_cleaned = df.head(1000)


# Filter out invalid coordinates (0,0 is a common placeholder)
df_cleaned = df_cleaned[~((df_cleaned['reclat'] == 0) & (df_cleaned['reclong'] == 0))]

# Convert 'year' to a numeric type, coercing errors, then drop rows with NaN in 'year'\ndf_cleaned['year'] = pd.to_numeric(df_cleaned['year'], errors='coerce')\ndf_cleaned.dropna(subset=['year'], inplace=True)
df_cleaned['Year'] = df_cleaned['Year'].astype(int)

ValueError: invalid literal for int() with base 10: 'DateObject[{1880}, "Year", "Gregorian", -5.]'

In [28]:
df_cleaned['mass_kg'] = df_cleaned['Mass'] / 1000

In [29]:
df_cleaned.info()
df_cleaned.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            0 non-null      object 
 1   ID              0 non-null      int64  
 2   NameType        0 non-null      object 
 3   Classification  0 non-null      object 
 4   Mass            0 non-null      float64
 5   Fall            0 non-null      object 
 6   Year            0 non-null      int64  
 7   Coordinates     0 non-null      object 
 8   reclat          0 non-null      float64
 9   reclong         0 non-null      float64
 10  mass_kg         0 non-null      float64
dtypes: float64(4), int64(2), object(5)
memory usage: 0.0+ bytes


,ID,Mass,Year,reclat,reclong,mass_kg
count,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN
